In [6]:
import pandas as pd
import numpy as np

orders_files = [
    "data/orders_2024_01.csv",
    "data/orders_2024_02.csv",
    "data/orders_2024_03.csv"
]

df_orders = pd.concat(
    [pd.read_csv(f) for f in orders_files],
    ignore_index=True
)

df_orders["price"] = pd.to_numeric(
    df_orders["price"], errors="coerce"
).astype("float32")

df_orders["quantity"] = pd.to_numeric(
    df_orders["quantity"], errors="coerce"
)

df_orders = df_orders.dropna(
    subset=["user_id", "order_id", "price", "quantity"]
)

print(df_orders.head())
print(df_orders.info())

   order_id  user_id       order_datetime  product_id         price  quantity  \
0         1     1044  2024-01-03 18:39:26          26  14038.150391         1   
1         2     1079  2024-01-03 17:45:43          16   9172.599609         1   
2         3     1041  2024-01-06 22:46:38          25   9035.259766         2   
3         4     1083  2024-01-08 21:03:51           9  15234.330078         3   
4         5     1068  2024-01-03 23:26:53          24   4134.379883         2   

     status  
0  refunded  
1      paid  
2      paid  
3      paid  
4      paid  
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        15000 non-null  int64  
 1   user_id         15000 non-null  int64  
 2   order_datetime  15000 non-null  str    
 3   product_id      15000 non-null  int64  
 4   price           15000 non-null  float32
 5   quantity    

In [7]:
# Задание 1
user_avg_price = df_orders.groupby("user_id").agg(
    avg_price=("price", "mean")
)

order_quantity = df_orders.groupby(
    ["user_id", "order_id"]
).agg(
    items_count=("quantity", "sum")
)

user_avg_quantity = order_quantity.groupby(
    "user_id"
).agg(
    avg_items_per_order=("items_count", "mean")
)

user_features = user_avg_price.join(
    user_avg_quantity
)

print(user_features.head())

            avg_price  avg_items_per_order
user_id                                   
1001      9645.921875             2.564972
1002     10042.691406             2.284553
1003     10323.160156             2.493590
1004      9977.171875             2.585034
1005      9544.485352             2.453901


In [8]:
# Задание 2
features = user_features[
    ["avg_price", "avg_items_per_order"]
]

features_z = (
    features - features.mean()
) / features.std(ddof=0)

print(features_z.head())

         avg_price  avg_items_per_order
user_id                                
1001     -1.158585             0.738470
1002     -0.335493            -2.648359
1003      0.246335            -0.123664
1004     -0.471412             0.980777
1005     -1.369013            -0.603018


In [9]:
# Задание 3
max_user = features_z["avg_price"].idxmax()
max_value = features_z["avg_price"].max()

min_user = features_z["avg_price"].idxmin()
min_value = features_z["avg_price"].min()

print("Пользователь с самой высокой средней ценой:")
print("ID:", max_user)
print("Z-значение:", round(max_value, 3))

print("\nПользователь с самой низкой средней ценой:")
print("ID:", min_user)
print("Z-значение:", round(min_value, 3))

Пользователь с самой высокой средней ценой:
ID: 1040
Z-значение: 2.512

Пользователь с самой низкой средней ценой:
ID: 1063
Z-значение: -2.031
